# Decoder-only LoRMA + target-token BiLSTM


In [1]:
%pip install -q huggingface_hub tokenizers scikit-learn tqdm

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, json, math, random, gc, tempfile, zipfile, uuid, re, hashlib
from pathlib import Path
from collections import Counter
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer
from huggingface_hub import hf_hub_download, HfFileSystem
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
from tqdm.auto import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42
DATA_REPO = 'Exploration-Lab/CS-781-Capstone'
BUCKET_ROOT = 'hf://buckets/A-240031/legal-lm-phase1-bucket'
PHASE1_TAG = 'decoder_126M'
MODEL_PATH = 'checkpoints/decoder_126M/model.pt'
TOKENIZER_PATH = 'tokenizer_126M_decoder/tokenizer.json'
MODEL_CONFIG = dict(d_model=1024, n_heads=8, n_kv_heads=4, n_layers=8, d_ff=2816, max_seq_len=512)
MAX_SEQ_LEN = 512
# Number of previous AND following sentences from the same document.
# 0 = target only; 1 = one on each side; 2 = two on each side.
CONTEXT_SENTENCES = 2
assert isinstance(CONTEXT_SENTENCES, int) and CONTEXT_SENTENCES >= 0
assert 0 < MAX_SEQ_LEN <= MODEL_CONFIG['max_seq_len']
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
LR = 2e-4
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
NUM_EPOCHS = 20
PATIENCE = 5
MIN_DELTA = 0.001
ADAPTER_CONFIG = dict(r=16, alpha=32, dropout=0.05, do_pi=True)
HEAD_DROPOUT = 0.1
RR_POOLING = 'bilstm'
ATTENTION_POOL_HIDDEN_DIM = 128
LSTM_HIDDEN_SIZE = 256  # Per direction; concatenated output dimension = 512.
LSTM_NUM_LAYERS = 1
LSTM_DROPOUT = 0.1  # Between LSTM layers only when NUM_LAYERS > 1.
UNFREEZE_LAST_N_BLOCKS = 0
assert 0 <= UNFREEZE_LAST_N_BLOCKS <= MODEL_CONFIG['n_layers']
UPLOAD_CHECKPOINTS = True
TECHNIQUE = 'lorma'

# All loading/saving locations are controlled here.
# Paths below are relative to BUCKET_ROOT; an absolute hf:// path also works.
SAVE_TAG = None  # None generates a unique tag; or enter your own run tag.
SUBMISSION_ZIP_NAME = None  # None uses SAVE_TAG + '.zip'; or enter a filename.
CHECKPOINT_PREFIX = 'checkpoints'
RESUME = False  # True resumes SAVE_TAG; set SAVE_TAG to the existing latest tag.
WORK_ROOT = Path('/kaggle/working/rr_runs') if Path('/kaggle/working').exists() else Path.cwd() / 'rr_runs'

if RESUME and SAVE_TAG is None:
    raise ValueError('To resume, set SAVE_TAG to the existing run tag.')
if SAVE_TAG is None:
    SAVE_TAG = f'rr_{PHASE1_TAG}_joint_{TECHNIQUE}_ctx{CONTEXT_SENTENCES}_{RR_POOLING}_partial{UNFREEZE_LAST_N_BLOCKS}'
if not isinstance(SAVE_TAG, str) or not re.fullmatch(r'[A-Za-z0-9_.-]+', SAVE_TAG) or SAVE_TAG in {'.', '..'}:
    raise ValueError('SAVE_TAG must be a single name containing letters, numbers, _, - or .')
BEST_TAG = SAVE_TAG + '_best'
if SUBMISSION_ZIP_NAME is None:
    SUBMISSION_ZIP_NAME = SAVE_TAG + '.zip'
if Path(SUBMISSION_ZIP_NAME).name != SUBMISSION_ZIP_NAME or not SUBMISSION_ZIP_NAME.endswith('.zip'):
    raise ValueError('SUBMISSION_ZIP_NAME must be a filename ending in .zip.')
WORK_DIR = WORK_ROOT / SAVE_TAG
WORK_DIR.mkdir(parents=True, exist_ok=True)

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF')
    except Exception:
        pass
fs = HfFileSystem(token=HF_TOKEN)

def seed_everything(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
seed_everything(SEED)
print('Device:', DEVICE)
def bucket_path(path):
    if path.startswith('hf://'):
        return path
    return BUCKET_ROOT.rstrip('/') + '/' + path.lstrip('/')

def checkpoint_remote_path(tag):
    return bucket_path(f"{CHECKPOINT_PREFIX.strip('/')}/{tag}/checkpoint.pt")

def download_bucket_file(remote_path):
    # Identity-specific cache directory; refresh the bytes on each call.
    remote = bucket_path(remote_path)
    cache_key = hashlib.sha256(remote.encode()).hexdigest()[:20]
    local = WORK_ROOT / '_inputs' / cache_key / remote.rsplit('/', 1)[-1]
    local.parent.mkdir(parents=True, exist_ok=True)
    temporary = local.with_name(local.name + '.download')
    fs.get_file(remote, str(temporary))
    os.replace(temporary, local)
    return local

def assert_run_destination():
    if RESUME:
        return
    for tag in [SAVE_TAG, BEST_TAG]:
        local = WORK_DIR / 'checkpoints' / tag / 'checkpoint.pt'
        if local.exists() or (UPLOAD_CHECKPOINTS and fs.exists(checkpoint_remote_path(tag))):
            raise FileExistsError(
                f'Checkpoint already exists for {tag}. Choose a new SAVE_TAG, '
                'or set RESUME=True with the original latest SAVE_TAG.'
            )
    manifest = dict(save_tag=SAVE_TAG, best_tag=BEST_TAG, technique=TECHNIQUE,
                    phase1_tag=PHASE1_TAG, model_path=MODEL_PATH,
                    tokenizer_path=TOKENIZER_PATH, submission_zip=SUBMISSION_ZIP_NAME,
                    context_sentences=CONTEXT_SENTENCES, input_format='neighbors_then_target_eos_v1',
                    truncation='target_first_nearest_context_balanced', pooling=RR_POOLING,
                    unfreeze_last_n_blocks=UNFREEZE_LAST_N_BLOCKS,
                    attention_pool_hidden_dim=ATTENTION_POOL_HIDDEN_DIM,
                    lstm_hidden_size=LSTM_HIDDEN_SIZE, lstm_num_layers=LSTM_NUM_LAYERS,
                    lstm_dropout=LSTM_DROPOUT)
    with open(WORK_DIR / 'run_config.json', 'w') as f:
        json.dump(manifest, f, indent=2)

print('Phase-1 model:', bucket_path(MODEL_PATH))
print('Tokenizer:', bucket_path(TOKENIZER_PATH))
print('Latest checkpoint:', checkpoint_remote_path(SAVE_TAG))
print('Best checkpoint:', checkpoint_remote_path(BEST_TAG))
print('Submission ZIP:', WORK_DIR / SUBMISSION_ZIP_NAME)

print('Neighbor sentences on each side:', CONTEXT_SENTENCES)


Device: cuda
Phase-1 model: hf://buckets/A-240031/legal-lm-phase1-bucket/checkpoints/decoder_126M/model.pt
Tokenizer: hf://buckets/A-240031/legal-lm-phase1-bucket/tokenizer_126M_decoder/tokenizer.json
Latest checkpoint: hf://buckets/A-240031/legal-lm-phase1-bucket/checkpoints/rr_decoder_126M_joint_lorma_ctx2_bilstm_partial0/checkpoint.pt
Best checkpoint: hf://buckets/A-240031/legal-lm-phase1-bucket/checkpoints/rr_decoder_126M_joint_lorma_ctx2_bilstm_partial0_best/checkpoint.pt
Submission ZIP: /kaggle/working/rr_runs/rr_decoder_126M_joint_lorma_ctx2_bilstm_partial0/rr_decoder_126M_joint_lorma_ctx2_bilstm_partial0.zip
Neighbor sentences on each side: 2


In [3]:
local_tokenizer = download_bucket_file(TOKENIZER_PATH)
tokenizer = Tokenizer.from_file(str(local_tokenizer))
tokenizer.no_padding()
tokenizer.no_truncation()
VOCAB_SIZE = tokenizer.get_vocab_size()
PAD_ID = tokenizer.token_to_id('<|pad|>')
EMPTY_ID = tokenizer.token_to_id('<|eos|>')
assert PAD_ID is not None and EMPTY_ID is not None

EXPECTED_LABELS = ['Fact', 'Issue', 'ArgumentPetitioner', 'ArgumentRespondent', 'Statute', 'Dissent',
                   'PrecedentReliedUpon', 'PrecedentNotReliedUpon', 'PrecedentOverruled',
                   'RulingByLowerCourt', 'RatioOfTheDecision', 'RulingByPresentCourt', 'None']

def download_json(filename):
    path = hf_hub_download(repo_id=DATA_REPO, filename=filename, repo_type='dataset', token=HF_TOKEN)
    with open(path, encoding='utf-8') as f:
        return json.load(f)

vocab = download_json('Phase_3_RR/Train/label_vocab.json')
if isinstance(vocab, list) and all(isinstance(x, str) for x in vocab):
    LABELS = vocab
elif isinstance(vocab, dict) and set(vocab) == set(EXPECTED_LABELS):
    LABELS = sorted(vocab, key=lambda x: int(vocab[x])) if all(isinstance(v, int) for v in vocab.values()) else list(vocab)
elif isinstance(vocab, dict) and set(vocab.values()) == set(EXPECTED_LABELS):
    LABELS = [vocab[k] for k in sorted(vocab, key=lambda x: int(x))]
else:
    raise ValueError(f'Unexpected label_vocab format: {vocab!r}')
assert len(LABELS) == 13 and set(LABELS) == set(EXPECTED_LABELS)
label2id = {label: i for i, label in enumerate(LABELS)}
id2label = dict(enumerate(LABELS))

def normalize_documents(raw, labelled):
    if isinstance(raw, list):
        docs = raw
    elif isinstance(raw, dict) and 'id' in raw and 'text' in raw:
        docs = [raw]
    elif isinstance(raw, dict):
        docs = []
        for doc_id, value in raw.items():
            if isinstance(value, list) and not labelled:
                doc = {'id': doc_id, 'text': value}
            elif isinstance(value, dict):
                doc = dict(value)
                doc.setdefault('id', doc_id)
                assert doc['id'] == doc_id, 'Document ID mismatch'
            else:
                raise ValueError(f'Unexpected value for document {doc_id!r}: {type(value).__name__}')
            docs.append(doc)
    else:
        raise ValueError('Unsupported document container')
    result, seen = [], set()
    for doc in docs:
        doc = dict(doc)
        assert isinstance(doc['id'], str), 'Document IDs must be strings'
        assert doc['id'] not in seen, f"Duplicate ID: {doc['id']}"
        seen.add(doc['id'])
        assert isinstance(doc['text'], list) and all(isinstance(s, str) for s in doc['text'])
        if labelled:
            assert len(doc['labels']) == len(doc['text'])
            assert all(label in label2id for label in doc['labels'])
        if not labelled:
            doc.pop("labels", None)
        result.append(doc)
    return result

data = {}
for domain in ['CL', 'IT']:
    data[domain] = {}
    for split in ['train', 'dev', 'test']:
        folder = 'Test' if split == 'test' else 'Train'
        data[domain][split] = normalize_documents(
            download_json(f'Phase_3_RR/{folder}/{domain}_{split}.json'), split != 'test')
    assert not ({d['id'] for d in data[domain]['train']} & {d['id'] for d in data[domain]['dev']})
    for split, docs in data[domain].items():
        print(domain, split, 'documents:', len(docs), 'sentences:', sum(len(d['text']) for d in docs))
    print('Training label counts:', dict(Counter(l for d in data[domain]['train'] for l in d['labels'])))
assert not ({d['id'] for d in data['CL']['test']} & {d['id'] for d in data['IT']['test']}), 'Test IDs collide across domains'
print('Label mapping:', label2id)

(…)et/tokenizer_126M_decoder/tokenizer.json:   0%|          | 0.00/2.47M [00:00<?, ?B/s]

label_vocab.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

CL_dev.json:   0%|          | 0.00/336k [00:00<?, ?B/s]

CL_test.json:   0%|          | 0.00/281k [00:00<?, ?B/s]

CL train documents: 40 sentences: 10593
CL dev documents: 5 sentences: 1396
CL test documents: 5 sentences: 1339
Training label counts: {'Fact': 2979, 'ArgumentPetitioner': 976, 'ArgumentRespondent': 996, 'RatioOfTheDecision': 2856, 'RulingByPresentCourt': 113, 'RulingByLowerCourt': 438, 'None': 53, 'PrecedentNotReliedUpon': 209, 'PrecedentReliedUpon': 1305, 'Statute': 367, 'Issue': 17, 'Dissent': 284}


IT_train.json:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

IT_dev.json:   0%|          | 0.00/165k [00:00<?, ?B/s]

IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

IT train documents: 40 sentences: 6252
IT dev documents: 5 sentences: 746
IT test documents: 5 sentences: 858
Training label counts: {'Fact': 1119, 'RulingByLowerCourt': 436, 'ArgumentPetitioner': 637, 'Issue': 90, 'Statute': 732, 'RatioOfTheDecision': 1445, 'RulingByPresentCourt': 194, 'PrecedentReliedUpon': 640, 'ArgumentRespondent': 352, 'PrecedentNotReliedUpon': 360, 'None': 244, 'PrecedentOverruled': 3}
Label mapping: {'Fact': 0, 'Issue': 1, 'ArgumentPetitioner': 2, 'ArgumentRespondent': 3, 'Statute': 4, 'Dissent': 5, 'PrecedentReliedUpon': 6, 'PrecedentNotReliedUpon': 7, 'PrecedentOverruled': 8, 'RulingByLowerCourt': 9, 'RatioOfTheDecision': 10, 'RulingByPresentCourt': 11, 'None': 12}


In [4]:
tokenizer.get_vocab_size()

33467

In [5]:
def build_context_input(sentence_tokens, index):
    """Use neighboring text only; never access neighboring labels.

    Reserve target + EOS first. Allocate remaining tokens among nearby
    context sentences. Previous sentences keep their ends; following
    sentences keep their beginnings. Context stays in document order,
    but the target always appears last for causal attention.
    """
    target = sentence_tokens[index]
    if CONTEXT_SENTENCES == 0:
        ids = target[:MAX_SEQ_LEN - 1] + [EMPTY_ID]
        return ids, dict(target_truncated=len(target) > MAX_SEQ_LEN - 1,
                         context_truncated=False, neighbors_used=0,
                         target_start=0, target_end=max(1, len(ids) - 1))

    target_marker = tokenizer.encode(
        "Target sentence:\n", add_special_tokens=False
    ).ids
    target_capacity = MAX_SEQ_LEN - len(target_marker) - 1
    if target_capacity < 1:
        raise ValueError('MAX_SEQ_LEN is too small for the target marker and EOS.')
    kept_target = target[:target_capacity]
    budget = MAX_SEQ_LEN - len(target_marker) - len(kept_target) - 1

    candidates = []
    # Closest sentences get first access when there is very little space.
    for distance in range(1, min(CONTEXT_SENTENCES, len(sentence_tokens) - 1) + 1):
        for offset, side in [(-distance, 'Previous'), (distance, 'Following')]:
            neighbor_index = index + offset
            if not 0 <= neighbor_index < len(sentence_tokens):
                continue
            tokens = sentence_tokens[neighbor_index]
            if not tokens:
                continue
            marker = tokenizer.encode(
                f"{side} sentence ({offset:+d}):\n",
                add_special_tokens=False,
            ).ids
            candidates.append(dict(offset=offset, marker=marker, tokens=tokens))

    selected = []
    for candidate in candidates:
        # Reserve the marker and at least one token of neighbor content.
        cost = len(candidate['marker']) + 1
        if budget >= cost:
            selected.append(dict(**candidate, allocated=1))
            budget -= cost

    # Share spare tokens between neighbors, redistributing unused capacity.
    while budget > 0:
        progressed = False
        for candidate in selected:
            if candidate['allocated'] < len(candidate['tokens']):
                candidate['allocated'] += 1
                budget -= 1
                progressed = True
                if budget == 0:
                    break
        if not progressed:
            break

    ids = []
    # Previous context oldest -> nearest; following context nearest -> farthest.
    for candidate in sorted(selected, key=lambda c: c['offset']):
        count = candidate['allocated']
        tokens = candidate['tokens']
        kept = tokens[-count:] if candidate['offset'] < 0 else tokens[:count]
        ids.extend(candidate['marker'])
        ids.extend(kept)
    ids.extend(target_marker)
    target_start = len(ids)
    ids.extend(kept_target)
    target_end = len(ids)
    ids.append(EMPTY_ID)
    if target_start == target_end:
        # Empty targets have no lexical state; use their final EOS as fallback.
        target_start, target_end = len(ids) - 1, len(ids)
    assert 1 <= len(ids) <= MAX_SEQ_LEN

    context_truncated = (
        len(selected) < len(candidates)
        or any(c['allocated'] < len(c['tokens']) for c in selected)
    )
    return ids, dict(target_truncated=len(target) > len(kept_target),
                     context_truncated=context_truncated,
                     neighbors_used=len(selected),
                     target_start=target_start, target_end=target_end)


class RRSentenceDataset(Dataset):
    def __init__(self, documents, domain):
        self.rows = []
        stats = Counter()
        for doc in documents:
            # Tokenize once per sentence rather than repeatedly per neighbor.
            sentence_tokens = [
                tokenizer.encode(s, add_special_tokens=False).ids
                for s in doc['text']
            ]
            for index, tokens in enumerate(sentence_tokens):
                ids, input_stats = build_context_input(sentence_tokens, index)
                target_start = input_stats.pop('target_start')
                target_end = input_stats.pop('target_end')
                assert 0 <= target_start < target_end <= len(ids)
                stats['empty_targets'] += not tokens
                stats.update(input_stats)
                self.rows.append(dict(
                    input_ids=ids,
                    target_start=target_start, target_end=target_end,
                    doc_id=doc['id'],
                    sentence_index=index,
                    domain=domain,
                    label=label2id[doc['labels'][index]] if 'labels' in doc else -100,
                ))
        print(domain, 'examples:', len(self.rows),
              'context per side:', CONTEXT_SENTENCES, 'input stats:', dict(stats))

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        return self.rows[index]


def collate_rr(rows):
    lengths = torch.tensor([len(r['input_ids']) for r in rows], dtype=torch.long)
    ids = torch.full((len(rows), int(lengths.max())), PAD_ID, dtype=torch.long)
    for i, r in enumerate(rows):
        ids[i, :lengths[i]] = torch.tensor(r['input_ids'])
    return dict(input_ids=ids, lengths=lengths,
                target_starts=torch.tensor([r['target_start'] for r in rows], dtype=torch.long),
                target_ends=torch.tensor([r['target_end'] for r in rows], dtype=torch.long),
                labels=torch.tensor([r['label'] for r in rows]),
                doc_ids=[r['doc_id'] for r in rows], indices=[r['sentence_index'] for r in rows])

datasets = {domain: {split: RRSentenceDataset(docs, domain) for split, docs in splits.items()}
            for domain, splits in data.items()}
def make_loader(domain, split):
    return DataLoader(datasets[domain][split], batch_size=BATCH_SIZE, shuffle=(split == 'train'),
                      collate_fn=collate_rr, num_workers=0, pin_memory=(DEVICE == 'cuda'))
from torch.utils.data import ConcatDataset
datasets['JOINT'] = {split: ConcatDataset([datasets['CL'][split], datasets['IT'][split]]) for split in ['train', 'dev']}
assert len(datasets['JOINT']['train']) == len(datasets['CL']['train']) + len(datasets['IT']['train'])


CL examples: 10593 context per side: 2 input stats: {'empty_targets': 0, 'target_truncated': 3, 'context_truncated': 133, 'neighbors_used': 42120}
CL examples: 1396 context per side: 2 input stats: {'empty_targets': 0, 'target_truncated': 0, 'context_truncated': 18, 'neighbors_used': 5554}
CL examples: 1339 context per side: 2 input stats: {'empty_targets': 0, 'target_truncated': 1, 'context_truncated': 20, 'neighbors_used': 5322}
IT examples: 6252 context per side: 2 input stats: {'empty_targets': 0, 'target_truncated': 3, 'context_truncated': 159, 'neighbors_used': 24756}
IT examples: 746 context per side: 2 input stats: {'empty_targets': 0, 'target_truncated': 0, 'context_truncated': 6, 'neighbors_used': 2954}
IT examples: 858 context per side: 2 input stats: {'empty_targets': 0, 'target_truncated': 0, 'context_truncated': 20, 'neighbors_used': 3402}


In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# RoPE
# ============================================================

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len=512, base=10000):
        super().__init__()

        inv_freq = 1.0 / (
            base ** (torch.arange(0, head_dim, 2).float() / head_dim)
        )

        positions = torch.arange(max_seq_len).float()
        freqs = torch.outer(positions, inv_freq)

        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    def forward(self, q, k):
        # q, k: [B, H, T, D]
        T = q.size(2)

        cos = self.cos[:T].unsqueeze(0).unsqueeze(0)
        sin = self.sin[:T].unsqueeze(0).unsqueeze(0)

        q1, q2 = q[..., ::2], q[..., 1::2]
        k1, k2 = k[..., ::2], k[..., 1::2]

        q_rot = torch.stack(
            [q1 * cos - q2 * sin,
             q1 * sin + q2 * cos],
            dim=-1,
        ).flatten(-2)

        k_rot = torch.stack(
            [k1 * cos - k2 * sin,
             k1 * sin + k2 * cos],
            dim=-1,
        ).flatten(-2)

        return q_rot, k_rot


# ============================================================
# SwiGLU
# ============================================================

class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()

        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.down_proj(
            F.silu(self.gate_proj(x)) * self.up_proj(x)
        )


# ============================================================
# GQA + SDPA
# ============================================================

class GQAAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads):
        super().__init__()

        assert d_model % n_heads == 0
        assert n_heads % n_kv_heads == 0

        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_heads

        self.q_proj = nn.Linear(
            d_model,
            n_heads * self.head_dim,
            bias=False,
        )

        self.k_proj = nn.Linear(
            d_model,
            n_kv_heads * self.head_dim,
            bias=False,
        )

        self.v_proj = nn.Linear(
            d_model,
            n_kv_heads * self.head_dim,
            bias=False,
        )

        self.out_proj = nn.Linear(
            d_model,
            d_model,
            bias=False,
        )

        self.rope = RotaryEmbedding(
            self.head_dim,
            max_seq_len=512,
        )

    def forward(self, x):

        B, T, _ = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(
            B, T,
            self.n_heads,
            self.head_dim
        ).transpose(1, 2)

        k = k.view(
            B, T,
            self.n_kv_heads,
            self.head_dim
        ).transpose(1, 2)

        v = v.view(
            B, T,
            self.n_kv_heads,
            self.head_dim
        ).transpose(1, 2)

        q, k = self.rope(q, k)

        # Repeat KV heads for GQA
        repeat_factor = self.n_heads // self.n_kv_heads

        k = k.repeat_interleave(
            repeat_factor,
            dim=1,
        )

        v = v.repeat_interleave(
            repeat_factor,
            dim=1,
        )

        # PyTorch SDPA can dispatch to Flash Attention
        # / memory-efficient kernels automatically.
        attn_out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            is_causal=True,
        )

        attn_out = (
            attn_out
            .transpose(1, 2)
            .contiguous()
            .view(B, T, -1)
        )

        return self.out_proj(attn_out)


# ============================================================
# Decoder Block
# ============================================================

class DecoderBlock(nn.Module):
    def __init__(
        self,
        d_model,
        n_heads,
        n_kv_heads,
        d_ff,
    ):
        super().__init__()

        self.norm1 = nn.RMSNorm(d_model)
        self.attention = GQAAttention(
            d_model,
            n_heads,
            n_kv_heads,
        )

        self.norm2 = nn.RMSNorm(d_model)
        self.ffn = SwiGLU(
            d_model,
            d_ff,
        )

    def forward(self, x):

        x = x + self.attention(
            self.norm1(x)
        )

        x = x + self.ffn(
            self.norm2(x)
        )

        return x


# ============================================================
# Decoder-only Language Model
# ============================================================

class EfficientDecoderLM(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model=768,
        n_heads=12,
        n_kv_heads=4,
        n_layers=12,
        d_ff=3072,
        max_seq_len=512,
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.d_model = d_model

        self.token_embedding = nn.Embedding(                             
            vocab_size,
            d_model,
        )

        self.blocks = nn.ModuleList([
            DecoderBlock(
                d_model=d_model,
                n_heads=n_heads,
                n_kv_heads=n_kv_heads,
                d_ff=d_ff,
            )
            for _ in range(n_layers)
        ])

        self.final_norm = nn.RMSNorm(d_model)

        # Tied LM head
        self.lm_head = nn.Linear(
            d_model,
            vocab_size,
            bias=False,
        )

        self.lm_head.weight = self.token_embedding.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):

        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)

        elif isinstance(module, nn.Embedding):
            nn.init.xavier_uniform_(module.weight)

        elif isinstance(module, nn.RMSNorm):
            nn.init.ones_(module.weight)

    def forward(self, input_ids):

        x = self.token_embedding(input_ids)

        for block in self.blocks:
            x = block(x)

        x = self.final_norm(x)

        return self.lm_head(x)

In [7]:
class LoRMALinear(nn.Module):
    def __init__(self, original_linear, r=16, alpha=32, dropout=0.05, do_pi=True):
        super().__init__()
        self.original_linear = original_linear
        self.in_features = original_linear.in_features
        self.out_features = original_linear.out_features
        self.scaling, self.do_pi = alpha / r, do_pi
        self.dropout = nn.Dropout(dropout)
        for p in original_linear.parameters(): p.requires_grad = False
        W = original_linear.weight
        self.A = nn.Parameter(W.new_empty(r, self.out_features))
        self.B = nn.Parameter(W.new_zeros(self.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
    def effective_weight(self):
        W = self.original_linear.weight
        delta = self.B @ self.A
        if self.do_pi:
            rows = torch.arange(self.out_features, device=W.device)[:, None]
            cols = torch.arange(self.out_features, device=W.device)[None, :]
            delta = delta.gather(1, (cols - rows) % self.out_features)
        return W + self.scaling * (delta @ W)
    def forward(self, x):
        W = self.original_linear.weight
        W_eff = self.effective_weight()
        return F.linear(x, W, self.original_linear.bias) + F.linear(self.dropout(x), W_eff - W)

class EfficientDecoderLMForRR(EfficientDecoderLM):
    def __init__(self, vocab_size, num_labels=13, head_dropout=0.1, **kwargs):
        super().__init__(vocab_size=vocab_size, **kwargs)
        del self.lm_head
        self.rr_dropout = nn.Dropout(head_dropout)
        if RR_POOLING == 'attention':
            self.rr_pooler = nn.Sequential(
                nn.Linear(self.d_model, ATTENTION_POOL_HIDDEN_DIM),
                nn.Tanh(),
                nn.Linear(ATTENTION_POOL_HIDDEN_DIM, 1, bias=False),
            )
            output_dim = self.d_model
        elif RR_POOLING == 'bilstm':
            self.rr_pooler = nn.LSTM(
                input_size=self.d_model,
                hidden_size=LSTM_HIDDEN_SIZE,
                num_layers=LSTM_NUM_LAYERS,
                batch_first=True,
                bidirectional=True,
                dropout=LSTM_DROPOUT if LSTM_NUM_LAYERS > 1 else 0.0,
            )
            output_dim = 2 * LSTM_HIDDEN_SIZE
        else:
            raise ValueError(f'Unknown RR_POOLING: {RR_POOLING}')
        self.rr_head = nn.Linear(output_dim, num_labels)

    def pool_target(self, hidden, lengths, target_starts, target_ends):
        starts = target_starts.to(hidden.device)
        ends = target_ends.to(hidden.device)
        lengths = lengths.to(hidden.device)
        assert starts.shape == ends.shape == lengths.shape == (hidden.size(0),)
        assert bool(((starts >= 0) & (ends > starts) & (ends <= lengths)).all())
        counts = ends - starts
        max_target_len = int(counts.max().item())
        offsets = torch.arange(max_target_len, device=hidden.device)[None, :]
        mask = offsets < counts[:, None]
        positions = starts[:, None] + offsets
        positions = positions.clamp(max=hidden.size(1) - 1)
        target = hidden.gather(
            1, positions[..., None].expand(-1, -1, hidden.size(-1))
        )
        target = target.masked_fill(~mask[..., None], 0.0)
        if RR_POOLING == 'attention':
            scores = self.rr_pooler(target).squeeze(-1)
            scores = scores.masked_fill(~mask, float('-inf'))
            weights = torch.softmax(scores, dim=1)
            return (weights[..., None] * target).sum(dim=1)
        # Packing prevents padded target positions from affecting either direction.
        packed = nn.utils.rnn.pack_padded_sequence(
            target, counts.detach().cpu(), batch_first=True, enforce_sorted=False
        )
        _, (h_n, _) = self.rr_pooler(packed)
        return torch.cat([h_n[-2], h_n[-1]], dim=-1)

    def forward(self, input_ids, lengths, target_starts, target_ends):
        x = self.token_embedding(input_ids)
        for block in self.blocks:
            x = block(x)  # Dense decoder-only block returns a tensor.
        x = self.final_norm(x)
        pooled = self.pool_target(x, lengths, target_starts, target_ends)
        return self.rr_head(self.rr_dropout(pooled))


def forward_rr_batch(model, batch):
    return model(
        batch['input_ids'].to(DEVICE),
        batch['lengths'].to(DEVICE),
        batch['target_starts'].to(DEVICE),
        batch['target_ends'].to(DEVICE),
    )

phase1_path = download_bucket_file(MODEL_PATH)

def build_rr_model():
    model = EfficientDecoderLMForRR(VOCAB_SIZE, head_dropout=HEAD_DROPOUT, **MODEL_CONFIG)
    checkpoint = torch.load(phase1_path, map_location='cpu', weights_only=False)
    if 'model_state_dict' in checkpoint:
        state = checkpoint['model_state_dict']
    elif 'model' in checkpoint:
        state = checkpoint['model']
    elif 'state_dict' in checkpoint:
        state = checkpoint['state_dict']
    elif all(isinstance(v, torch.Tensor) for v in checkpoint.values()):
        state = checkpoint
    else:
        raise ValueError(f'Cannot identify model weights. Checkpoint keys: {list(checkpoint)}')
    state = dict(state)
    state.pop('lm_head.weight', None)
    missing, unexpected = model.load_state_dict(state, strict=False)
    expected_new = {k for k in model.state_dict() if k.startswith(('rr_head.', 'rr_pooler.'))}
    assert set(missing) == expected_new and not unexpected, (missing, unexpected)
    del state, checkpoint
    for p in model.parameters(): p.requires_grad = False
    for block in model.blocks:
        for name in ['q_proj', 'k_proj', 'v_proj', 'out_proj']:
            setattr(block.attention, name, LoRMALinear(getattr(block.attention, name), **ADAPTER_CONFIG))
    if UNFREEZE_LAST_N_BLOCKS:
        for block in model.blocks[-UNFREEZE_LAST_N_BLOCKS:]:
            for p in block.parameters():
                p.requires_grad = True
    for module in [model.rr_pooler, model.rr_head]:
        for p in module.parameters():
            p.requires_grad = True
    model = model.to(DEVICE)
    print('Total/trainable parameters:', sum(p.numel() for p in model.parameters()),
          sum(p.numel() for p in model.parameters() if p.requires_grad))
    return model

(…)bucket/checkpoints/decoder_126M/model.pt:   0%|          | 0.00/515M [00:00<?, ?B/s]

In [8]:
# CPU smoke checks; preserve RNG state so checks do not alter training initialization.
with torch.random.fork_rng(devices=[]):
    torch.manual_seed(123)
    small = EfficientDecoderLMForRR(
        32, d_model=16, n_heads=4, n_kv_heads=2,
        n_layers=4, d_ff=32, head_dropout=0,
    )
    for p in small.parameters():
        p.requires_grad = False
    for block in small.blocks:
        for name in ['q_proj', 'k_proj', 'v_proj', 'out_proj']:
            setattr(block.attention, name, LoRMALinear(
                getattr(block.attention, name), r=4, alpha=8, dropout=0,
            ))
    for block in small.blocks[-2:]:
        for p in block.parameters():
            p.requires_grad = True
    for module in [small.rr_pooler, small.rr_head]:
        for p in module.parameters():
            p.requires_grad = True
    assert not small.blocks[0].norm1.weight.requires_grad
    assert small.blocks[-1].norm1.weight.requires_grad
    assert small.blocks[0].attention.q_proj.B.requires_grad
    ids = torch.tensor([[1, 2, 3, 4, 5, 6], [7, 8, 9, 0, 0, 0]])
    lengths = torch.tensor([6, 3])
    starts = torch.tensor([2, 1])
    ends = torch.tensor([5, 2])
    small.eval()
    with torch.no_grad():
        logits = small(ids, lengths, starts, ends)
        one = small(ids[1:2, :3], lengths[1:2], starts[1:2], ends[1:2])
        torch.testing.assert_close(logits[1:2], one, atol=1e-5, rtol=1e-4)
        hidden = torch.randn(2, 6, 16)
        a = small.pool_target(hidden, lengths, starts, ends)
        altered = hidden.clone()
        for b in range(2):
            altered[b, :starts[b]] = 1000
            altered[b, ends[b]:] = -1000
        b = small.pool_target(altered, lengths, starts, ends)
        torch.testing.assert_close(a, b)
    assert logits.shape == (2, 13) and torch.isfinite(logits).all()
    F.cross_entropy(small(ids, lengths, starts, ends), torch.tensor([0, 1])).backward()
    assert any(p.grad is not None and p.grad.abs().sum() > 0 for p in small.rr_pooler.parameters())
    assert small.blocks[-1].norm1.weight.grad is not None
    assert small.blocks[0].norm1.weight.grad is None
    with tempfile.TemporaryDirectory() as folder:
        path = Path(folder) / 'smoke.pt'
        torch.save(small.state_dict(), path)
        small.load_state_dict(torch.load(path, map_location='cpu', weights_only=True), strict=True)
    del small, ids, lengths, starts, ends, logits, one, hidden, altered, a, b
print('Passed target masking, padding, pooling gradient, partial unfreezing and checkpoint checks')


Passed target masking, padding, pooling gradient, partial unfreezing and checkpoint checks


In [9]:
def experiment_metadata(domain):
    return dict(domain=domain, phase1_tag=PHASE1_TAG, model_path=MODEL_PATH, tokenizer_path=TOKENIZER_PATH,
                labels=LABELS, model_config=MODEL_CONFIG, adapter_config=ADAPTER_CONFIG,
                max_seq_len=MAX_SEQ_LEN, pooling=RR_POOLING, context='neighbors_then_target_eos_v1',
                context_sentences=CONTEXT_SENTENCES,
                unfreeze_last_n_blocks=UNFREEZE_LAST_N_BLOCKS,
                pooling_config=(dict(hidden_dim=ATTENTION_POOL_HIDDEN_DIM) if RR_POOLING == 'attention'
                                else dict(hidden_size=LSTM_HIDDEN_SIZE, num_layers=LSTM_NUM_LAYERS,
                                          recurrent_dropout=LSTM_DROPOUT, output_dropout=HEAD_DROPOUT)),
                empty_target_fallback='eos',
                truncation='target_first_nearest_context_balanced',
                technique='LoRMA_identity_init', head_dropout=HEAD_DROPOUT,
                seed=SEED, batch_size=BATCH_SIZE, grad_accum_steps=GRAD_ACCUM_STEPS,
                lr=LR, weight_decay=WEIGHT_DECAY, grad_clip=GRAD_CLIP,
                patience=PATIENCE, min_delta=MIN_DELTA)

def save_rr_checkpoint(tag, model, optimizer, epoch, extra, domain):
    folder = WORK_DIR / 'checkpoints' / tag
    folder.mkdir(parents=True, exist_ok=True)
    rng = {'python': random.getstate(), 'torch': torch.get_rng_state()}
    if torch.cuda.is_available(): rng['cuda'] = torch.cuda.get_rng_state_all()
    payload = dict(model=model.state_dict(), optimizer=optimizer.state_dict(), rng=rng,
                   epoch=epoch, extra=extra, metadata=experiment_metadata(domain))
    tmp = folder / 'checkpoint.tmp'
    torch.save(payload, tmp)
    os.replace(tmp, folder / 'checkpoint.pt')
    if UPLOAD_CHECKPOINTS:
        fs.put_file(str(folder / 'checkpoint.pt'), checkpoint_remote_path(tag))

def load_rr_checkpoint(tag, model, domain, optimizer=None, restore_rng=False):
    path = WORK_DIR / 'checkpoints' / tag / 'checkpoint.pt'
    if not path.exists() or restore_rng:
        if UPLOAD_CHECKPOINTS:
            path = download_bucket_file(checkpoint_remote_path(tag))
        elif not path.exists():
            raise FileNotFoundError(path)
    payload = torch.load(path, map_location='cpu', weights_only=False)  # own trusted checkpoints only
    assert payload['metadata'] == experiment_metadata(domain), 'Checkpoint configuration mismatch'
    model.load_state_dict(payload['model'])
    if optimizer is not None: optimizer.load_state_dict(payload['optimizer'])
    if restore_rng:
        random.setstate(payload['rng']['python'])
        torch.set_rng_state(payload['rng']['torch'])
        if torch.cuda.is_available() and 'cuda' in payload['rng']:
            torch.cuda.set_rng_state_all(payload['rng']['cuda'])
    return payload

@torch.no_grad()
def evaluate_rr(model, loader):
    model.eval()
    targets, predictions = [], []
    total_loss = total_n = 0
    for batch in loader:
        y = batch['labels'].to(DEVICE)
        logits = forward_rr_batch(model, batch)
        total_loss += F.cross_entropy(logits, y, reduction='sum').item()
        total_n += len(y)
        targets.extend(y.cpu().tolist())
        predictions.extend(logits.argmax(-1).cpu().tolist())
    assert total_n > 0, 'Development set has no sentences'
    p, r, f, support = precision_recall_fscore_support(targets, predictions, labels=list(range(13)), zero_division=0)
    return dict(loss=total_loss / total_n, macro_f1=float(f.mean()), macro_precision=float(p.mean()),
                macro_recall=float(r.mean()), per_label={label: dict(precision=float(p[i]), recall=float(r[i]),
                f1=float(f[i]), support=int(support[i])) for i, label in enumerate(LABELS)},
                confusion_matrix=confusion_matrix(targets, predictions, labels=list(range(13))).tolist())

def evaluate_joint(model):
    scores = {d: evaluate_rr(model, make_loader(d, 'dev')) for d in ['CL', 'IT']}
    scores['pooled'] = evaluate_rr(model, make_loader('JOINT', 'dev'))
    scores['selection_macro_f1'] = (scores['CL']['macro_f1'] + scores['IT']['macro_f1']) / 2
    return scores


In [10]:
def train_rr(domain='JOINT'):
    assert domain == 'JOINT'
    assert_run_destination()
    seed_everything(SEED)
    model = build_rr_model()
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)
    tag = SAVE_TAG
    best_tag = BEST_TAG
    start = 0
    extra = dict(best_f1=-1.0, patience_reference=-1.0, stale=0, history=[], best_tag=best_tag)
    if RESUME:
        checkpoint = load_rr_checkpoint(SAVE_TAG, model, domain, optimizer, restore_rng=True)
        start, extra = checkpoint['epoch'], checkpoint['extra']
        best_tag = extra['best_tag']
        del checkpoint
    train_loader, dev_loader = make_loader(domain, 'train'), make_loader(domain, 'dev')
    assert len(train_loader) > 0
    for epoch in range(start, NUM_EPOCHS):
        if extra['stale'] >= PATIENCE: break
        model.train()
        optimizer.zero_grad(set_to_none=True)
        train_sum = train_n = group_n = 0
        for step, batch in enumerate(tqdm(train_loader, desc=f'{domain} epoch {epoch+1}')):
            y = batch['labels'].to(DEVICE)
            logits = forward_rr_batch(model, batch)
            loss_sum = F.cross_entropy(logits, y, reduction='sum')
            loss_sum.backward()
            group_n += len(y)
            train_n += len(y)
            train_sum += loss_sum.item()
            if (step + 1) % GRAD_ACCUM_STEPS == 0 or step + 1 == len(train_loader):
                for p in params:
                    if p.grad is not None: p.grad.div_(group_n)
                torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP)
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                group_n = 0
        metrics = evaluate_joint(model)
        f1 = metrics['selection_macro_f1']
        improved = f1 > extra['best_f1']
        if improved: extra['best_f1'] = f1
        if f1 > extra['patience_reference'] + MIN_DELTA:
            extra['patience_reference'], extra['stale'] = f1, 0
        else: extra['stale'] += 1
        extra['history'].append(dict(epoch=epoch+1, train_loss=train_sum/train_n, **metrics))
        print(f"Joint train loss={train_sum/train_n:.4f}, selection macro-F1={f1:.4f}, CL={metrics['CL']['macro_f1']:.4f}, IT={metrics['IT']['macro_f1']:.4f}, pooled={metrics['pooled']['macro_f1']:.4f}, patience={extra['stale']}/{PATIENCE}")
        if improved: save_rr_checkpoint(best_tag, model, optimizer, epoch+1, extra, domain)
        save_rr_checkpoint(tag, model, optimizer, epoch+1, extra, domain)
    load_rr_checkpoint(best_tag, model, domain)
    metrics = evaluate_joint(model)
    with open(WORK_DIR / f'{domain}_dev_metrics.json', 'w') as f: json.dump(metrics, f, indent=2)
    with open(WORK_DIR / f'{domain}_history.json', 'w') as f: json.dump(extra['history'], f, indent=2)
    print('Best development metrics:', metrics)
    del optimizer
    return model, extra['history']


In [11]:
@torch.no_grad()
def predict_rr(model, domain):
    model.eval()
    docs = data[domain]['test']
    predictions = {d['id']: [None] * len(d['text']) for d in docs}
    seen = set()
    for batch in tqdm(make_loader(domain, 'test'), desc=f'{domain} test predictions'):
        logits = forward_rr_batch(model, batch)
        for doc_id, index, predicted_id in zip(batch['doc_ids'], batch['indices'], logits.argmax(-1).cpu().tolist()):
            assert (doc_id, index) not in seen
            seen.add((doc_id, index))
            predictions[doc_id][index] = id2label[predicted_id]
    validate_predictions(predictions, docs)
    return predictions

def validate_predictions(predictions, documents):
    assert set(predictions) == {d['id'] for d in documents}
    for doc in documents:
        labels = predictions[doc['id']]
        assert len(labels) == len(doc['text']), doc['id']
        assert all(isinstance(l, str) and l in label2id for l in labels), doc['id']

joint_model, joint_history = train_rr()
cl_predictions = predict_rr(joint_model, 'CL')
it_predictions = predict_rr(joint_model, 'IT')
for domain, output in [('CL', cl_predictions), ('IT', it_predictions)]:
    with open(WORK_DIR / f'{domain}_prediction.json', 'w', encoding='utf-8') as f:
        json.dump(output, f, ensure_ascii=False, indent=2)
del joint_model
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()


Total/trainable parameters: 132078093 3418637


JOINT epoch 1:   0%|          | 0/4212 [00:00<?, ?it/s]

Joint train loss=1.1896, selection macro-F1=0.3371, CL=0.2741, IT=0.4001, pooled=0.3743, patience=0/5


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

... (32 outputs left out)

New Data Upload: |          |  0.00B /  0.00B            

Best development metrics: {'CL': {'loss': 2.2375815154302114, 'macro_f1': 0.2801781744341987, 'macro_precision': 0.2745874807116161, 'macro_recall': 0.30244383205156244, 'per_label': {'Fact': {'precision': 0.4864864864864865, 'recall': 0.6506024096385542, 'f1': 0.5567010309278351, 'support': 249}, 'Issue': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'support': 3}, 'ArgumentPetitioner': {'precision': 0.5338983050847458, 'recall': 0.36627906976744184, 'f1': 0.43448275862068964, 'support': 344}, 'ArgumentRespondent': {'precision': 0.3790849673202614, 'recall': 0.2457627118644068, 'f1': 0.2982005141388175, 'support': 236}, 'Statute': {'precision': 0.7727272727272727, 'recall': 0.8793103448275862, 'f1': 0.8225806451612904, 'support': 58}, 'Dissent': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'support': 0}, 'PrecedentReliedUpon': {'precision': 0.415929203539823, 'recall': 0.33098591549295775, 'f1': 0.3686274509803922, 'support': 142}, 'PrecedentNotReliedUpon': {'precision': 0.0, 'recall':

CL test predictions:   0%|          | 0/335 [00:00<?, ?it/s]

IT test predictions:   0%|          | 0/215 [00:00<?, ?it/s]

In [12]:
# Reload saved domain predictions so packaging can be rerun without retraining.
with open(WORK_DIR / 'CL_prediction.json', encoding='utf-8') as f: cl_predictions = json.load(f)
with open(WORK_DIR / 'IT_prediction.json', encoding='utf-8') as f: it_predictions = json.load(f)
assert not (cl_predictions.keys() & it_predictions.keys()), 'Cross-domain document ID collision'
predictions = {**cl_predictions, **it_predictions}
validate_predictions(predictions, data['CL']['test'] + data['IT']['test'])
json_path = WORK_DIR / 'prediction.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)
zip_path = WORK_DIR / SUBMISSION_ZIP_NAME
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(json_path, arcname='prediction.json')
print('JSON:', json_path)
print('ZIP:', zip_path)
print('Documents:', len(predictions), 'sentences:', sum(map(len, predictions.values())))

JSON: /kaggle/working/rr_runs/rr_decoder_126M_joint_lorma_ctx2_bilstm_partial0/prediction.json
ZIP: /kaggle/working/rr_runs/rr_decoder_126M_joint_lorma_ctx2_bilstm_partial0/rr_decoder_126M_joint_lorma_ctx2_bilstm_partial0.zip
Documents: 10 sentences: 2197


In [13]:
# Checkpoint conversion belongs in a separate preprocessing notebook.
